# 03 · Video features (the "cards")

The organisers' notebook 01 describes each video with **AlexNet** (an image network from 2012).
We use **CLIP** instead: a newer model that was trained to match images with text, so its
numbers capture *what is in the picture* (dog, face, car, crowd ...).

For each of the **1,102 videos** (1,000 training + 102 test):
1. take 8 evenly spaced frames from the 3 s clip,
2. turn each frame into 512 numbers with CLIP,
3. average the 8 → one card of 512 numbers per video.

The result is cached in `cache/clip_b32.npz`, so this only runs once (a few minutes on CPU).

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # repo root, so `mindreading` imports work
import numpy as np
import matplotlib.pyplot as plt
from mindreading.config import DATA_DIR, RESULTS_DIR, WEIGHTS_DIR, SUBMISSION_DIR
print('data folder:', DATA_DIR)

data folder: C:\Users\canbe\Desktop\eeg-mind-reading-challenge\eeg-mind-reading-challenge


In [2]:
from mindreading.video_features import extract_clip, lookup
feats = extract_clip()
print('video ids   :', feats['ids'][:3], '...', feats['ids'][-3:])
print('per frame   :', feats['frames'].shape, '(videos x frames x numbers)')
print('one per clip:', feats['mean'].shape, '(videos x numbers)')

video ids   : [1 2 3] ... [1100 1101 1102]
per frame   : (1102, 8, 512) (videos x frames x numbers)
one per clip: (1102, 512) (videos x numbers)


### Sanity check: do similar videos get similar cards?
For three test videos, list the training videos with the most similar card (cosine similarity, 1 = identical).
Open the video files to see whether the neighbours really look alike.

In [3]:
train_ids, test_ids = np.arange(1, 1001), np.arange(1001, 1103)
A, B = lookup(feats, test_ids), lookup(feats, train_ids)
sim = A @ B.T
for row in [0, 1, 2]:
    best = np.argsort(-sim[row])[:3]
    print(f'test video {test_ids[row]}: most similar training videos',
          [(int(train_ids[b]), round(float(sim[row, b]), 2)) for b in best])

test video 1001: most similar training videos [(415, 0.82), (725, 0.74), (782, 0.74)]
test video 1002: most similar training videos [(45, 0.92), (128, 0.9), (86, 0.9)]
test video 1003: most similar training videos [(880, 0.77), (103, 0.77), (301, 0.77)]
